# Score a Fixture VCF

This tutorial batch-scores a one-row VCF with a local FASTA and writes per-row checksum receipts. It uses the same deterministic fixture runtime as the single-variant notebook, so the output is fixture-smoke evidence only.

In [1]:
from __future__ import annotations

import contextlib
import io
import json
import os
import sys
from pathlib import Path
from tempfile import TemporaryDirectory


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "geno_lewm").exists():
            return candidate
    raise RuntimeError("run this notebook from inside the GenoLeWM checkout")


ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

The VCF path writes one score JSONL row and one checksum receipt JSONL row per scored alternate.

In [2]:
from examples.scoring_fixture import (
    build_fixture_runtime,
    make_fixture_model_dir,
    write_fixture_fasta,
    write_fixture_vcf,
)

workdir = TemporaryDirectory()
scratch = Path(workdir.name)
model_dir = make_fixture_model_dir(scratch / "model")
runtime = build_fixture_runtime(model_dir)
vcf_path = write_fixture_vcf(scratch / "input.vcf")
fasta_path = write_fixture_fasta(scratch / "ref.fa")
scores_path = scratch / "scores.jsonl"
receipts_path = scratch / "receipts.jsonl"

runtime.score_vcf(
    vcf_path,
    fasta_path,
    scores_path,
    receipt_path=receipts_path,
    progress=False,
)
score_lines = scores_path.read_text(encoding="utf-8").splitlines()
receipt_lines = receipts_path.read_text(encoding="utf-8").splitlines()
first_score = json.loads(score_lines[0])
summary = {
    "first_score": {
        "bucket_id": first_score["bucket_id"],
        "generated_by": first_score["generated_by"],
        "sigma_calibrated": round(first_score["sigma_calibrated"], 6),
        "sigma_raw": round(first_score["sigma_raw"], 6),
    },
    "receipt_rows": len(receipt_lines),
    "score_rows": len(score_lines),
}
print(json.dumps(summary, sort_keys=True))

{"first_score": {"bucket_id": "other|mid|none", "generated_by": "geno-lewm-score", "sigma_calibrated": 0.000161, "sigma_raw": 0.000115}, "receipt_rows": 1, "score_rows": 1}


The first receipt validates against the same manifest and local FASTA window used for scoring.

In [3]:
from examples.scoring_fixture import REFERENCE_FASTA_SEQUENCE, verify_cli_args
from geno_lewm.cli import verify as verify_cli

first_receipt_path = scratch / "first.receipt.json"
first_receipt_path.write_text(receipt_lines[0], encoding="utf-8")

verify_output = io.StringIO()
with contextlib.redirect_stdout(verify_output):
    rc = verify_cli.main(
        verify_cli_args(
            first_receipt_path,
            model_dir / "manifest.json",
            REFERENCE_FASTA_SEQUENCE,
        )
    )

print(f"first receipt validation exit code: {rc}")
print(f"verifier final line: {verify_output.getvalue().splitlines()[-1]}")
workdir.cleanup()

first receipt validation exit code: 0
verifier final line: ok
